In [1]:
import os, sys, time
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.append("..")

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

from src.chains import (
    build_llm, build_intent_chain, SMALL_MODEL,
    MestreResponse, parser, MESTRE_SYSTEM, chat_prompt, Intent,
)
from src.tools import search_technique, get_embeddings

llm = build_llm()                    # gpt-oss-120b: resposta do Mestre
llm_small = build_llm(SMALL_MODEL)   # gpt-oss-20b: resumo/extração (orçamento diário separado)
intent_chain = build_intent_chain(llm)

print(llm.invoke("ping").content[:20], "|", llm_small.invoke("ping").content[:20])
print(intent_chain.invoke({"question": "Me mostra um vídeo de kimura"}))


pong | pong
reasoning='The student explicitly asks to see a kimura video.' intent='video' search_term='kimura bjj'


## 3.4 Chatbot com ConversationBufferMemory

Baseline de memória de curto prazo: cada turno é guardado literalmente e reinjetado no prompt via `MessagesPlaceholder`. Teste: pergunta de acompanhamento que só faz sentido com o contexto dos turnos anteriores.


In [2]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="history", return_messages=True)

mestre_chat_prompt = ChatPromptTemplate.from_messages([
    ("system", MESTRE_SYSTEM),
    MessagesPlaceholder("history"),   # turnos anteriores entram aqui, como mensagens
    ("human", "{question}"),
]).partial(format_instructions=parser.get_format_instructions())

mestre_chat = mestre_chat_prompt | llm | parser


def to_text(resp: MestreResponse) -> str:
    """Versão curta da resposta para guardar na memória (economiza tokens)."""
    text = resp.clarifying_question or resp.context
    if resp.steps:
        text += " Steps: " + " | ".join(resp.steps)
    return text


def chat(question: str, mem=memory) -> MestreResponse:
    history = mem.load_memory_variables({})["history"]
    resp = mestre_chat.invoke({"question": question, "history": history})
    mem.save_context({"input": question}, {"output": to_text(resp)})
    return resp


C:\Users\Giovanni Azevedo\AppData\Local\Temp\ipykernel_15196\1221989365.py:3: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="history", return_messages=True)
D:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent\.venv\Lib\site-packages\pydantic\main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pyda

In [3]:
turns = [
    "Sou faixa branca, treino com kimono e estou na guarda fechada. Como quebro a postura dele?",
    "Ele prendeu meu braço no chão. O que eu faço?",
    "E se ele defender isso?",
]

for q in turns:
    resp = chat(q)
    print(f">>> {q}\n    level={resp.level} | {resp.clarifying_question or resp.context}")
    if resp.steps:
        print("    steps:", resp.steps[:2])
    print()
    time.sleep(8)  # 8000 tokens/min


>>> Sou faixa branca, treino com kimono e estou na guarda fechada. Como quebro a postura dele?
    level=BEGINNER | Você está na guarda fechurada e quer quebrar a postura do oponente para impedir que ele fique em pé e ataque.
    steps: ['Segure a lapela do kimono com a mão mais próxima da cabeça dele e o colarinho com a outra mão, puxando-o levemente para baixo.', 'Use as duas mãos para controlar o braço oposto do oponente, puxando-o para dentro da sua guarda.']

>>> Ele prendeu meu braço no chão. O que eu faço?
    level=BEGINNER | Você está na guarda fechada e o oponente prendeu seu braço no chão; o objetivo é escapar e recuperar a guarda.
    steps: ['Use a mão livre para segurar a cabeça ou o colarinho do oponente, puxando-a para baixo e diminuindo a postura dele.', 'Gire o quadril para o lado oposto ao braço preso, criando espaço entre seu braço e o chão.']

>>> E se ele defender isso?
    level=BEGINNER | Se o oponente bloquear a sua tentativa de escapar do braço preso, você pod

In [4]:
time.sleep(8)
resp_no_mem = (chat_prompt | llm | parser).invoke({"question": turns[-1]})
print("SEM memória:", resp_no_mem.level, "|", resp_no_mem.clarifying_question or resp_no_mem.context)


SEM memória: BEGINNER | Qual movimento ou posição você está tentando executar e em que situação (gi ou no‑gi, cima ou baixo)?


In [5]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")

history = memory.load_memory_variables({})["history"]
for m in history:
    print(f"{type(m).__name__:12} {m.content[:100]}")
print("\ntokens no histórico:", sum(len(enc.encode(m.content)) for m in history))


HumanMessage Sou faixa branca, treino com kimono e estou na guarda fechada. Como quebro a postura dele?
AIMessage    Você está na guarda fechurada e quer quebrar a postura do oponente para impedir que ele fique em pé 
HumanMessage Ele prendeu meu braço no chão. O que eu faço?
AIMessage    Você está na guarda fechada e o oponente prendeu seu braço no chão; o objetivo é escapar e recuperar
HumanMessage E se ele defender isso?
AIMessage    Se o oponente bloquear a sua tentativa de escapar do braço preso, você pode usar outra sequência par

tokens no histórico: 542


## 3.2 Janela deslizante, resumo e busca vetorial

Conversa de 8 turnos; o turno 1 tem um fato crítico (lesão no joelho esquerdo). Pergunta final depende dele. Comparação: buffer (baseline), janela (`trim_messages`), resumo + janela (`ConversationSummaryBufferMemory`) e memória vetorial (`VectorStoreRetrieverMemory`, FAISS).


In [ ]:
from langchain_core.messages import trim_messages, get_buffer_string
from langchain_classic.memory import ConversationSummaryBufferMemory, VectorStoreRetrieverMemory
from langchain_community.vectorstores import FAISS

script = [
    ("Tenho uma lesão no joelho esquerdo, então evito torcer essa perna.",
     "Entendido. Vamos priorizar técnicas que não forcem o joelho esquerdo."),
    ("Sou faixa azul e treino com kimono.",
     "Ótimo, com kimono as pegadas de gola e manga ajudam muito no controle."),
    ("Na guarda fechada, como quebro a postura?",
     "Puxe a gola com as duas mãos e traga os joelhos ao peito para trazer ele para baixo."),
    ("E depois de quebrar a postura?",
     "Controle a cabeça e um braço; daí você pode atacar o armlock ou buscar a raspagem."),
    ("Como faço o armlock?",
     "Controle o braço, gire o quadril, passe a perna por cima da cabeça e eleve o quadril."),
    ("E se ele puxar o braço?",
     "Mude para o triângulo: o braço dele saiu e a cabeça ficou dentro da sua guarda."),
    ("Como fecho o triângulo direito?",
     "Ajuste o ângulo, puxe o braço dele cruzando e trave a perna atrás do joelho."),
    ("Ele levantou e ficou em pé na minha guarda.",
     "Abra a guarda e passe para a guarda aberta, com os pés nos quadris ou gancho de De La Riva."),
]
final_question = "Que raspagem da guarda aberta é segura pra mim?"

def count_tokens(msgs) -> int:
    return sum(len(enc.encode(m.content)) for m in msgs)

# 1) buffer (baseline)
buffer_mem = ConversationBufferMemory(memory_key="history", return_messages=True)

# 2) resumo + janela: ChatGroq não conta tokens sozinho -> tiktoken
llm_small_counter = build_llm(SMALL_MODEL, custom_get_token_ids=enc.encode)
summary_mem = ConversationSummaryBufferMemory(
    llm=llm_small_counter, max_token_limit=150, memory_key="history", return_messages=True
)

# 3) vetorial: cada turno vira um documento; FAISS precisa de 1 texto para nascer
episodes = FAISS.from_texts(["(início da conversa)"], get_embeddings())
vector_mem = VectorStoreRetrieverMemory(
    retriever=episodes.as_retriever(search_kwargs={"k": 2}), memory_key="history", input_key="input"
)

for human, ai in script:
    for mem in (buffer_mem, summary_mem, vector_mem):
        mem.save_context({"input": human}, {"output": ai})
    time.sleep(2)  # summary chama o 20b quando passa do limite

print("ok:", len(buffer_mem.chat_memory.messages), "mensagens no buffer")


C:\Users\Giovanni Azevedo\AppData\Local\Temp\ipykernel_3096\3213470657.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
C:\Users\Giovanni Azevedo\AppData\Local\Temp\ipykernel_3096\3213470657.py:33: LangChainDeprecationWarning: The class `ConversationSummaryBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  summary_mem = ConversationSummaryBufferMemory(
d:\repositorios_projetos\giovanni_azevedo_ai-agent_langchain_pd\bjj-agent

ok: 16 mensagens no buffer


In [ ]:
full_history = buffer_mem.load_memory_variables({})["history"]

window = trim_messages(
    full_history, strategy="last", max_tokens=150,
    token_counter=count_tokens, start_on="human",   # nunca começa com resposta solta do AI
)

contexts = {
    "buffer": get_buffer_string(full_history),
    "janela": get_buffer_string(window),
    "resumo+janela": get_buffer_string(summary_mem.load_memory_variables({})["history"]),
    "vetorial": vector_mem.load_memory_variables({"input": final_question})["history"],
}

for name, text in contexts.items():
    print(f"===== {name}: {len(enc.encode(text))} tokens | menciona joelho: {'joelho' in text.lower()}")
    print(text[:400], "\n")


===== buffer: 288 tokens | menciona joelho: True
Human: Tenho uma lesão no joelho esquerdo, então evito torcer essa perna.
AI: Entendido. Vamos priorizar técnicas que não forcem o joelho esquerdo.
Human: Sou faixa azul e treino com kimono.
AI: Ótimo, com kimono as pegadas de gola e manga ajudam muito no controle.
Human: Na guarda fechada, como quebro a postura?
AI: Puxe a gola com as duas mãos e traga os joelhos ao peito para trazer ele para bai 

===== janela: 146 tokens | menciona joelho: True
Human: Como faço o armlock?
AI: Controle o braço, gire o quadril, passe a perna por cima da cabeça e eleve o quadril.
Human: E se ele puxar o braço?
AI: Mude para o triângulo: o braço dele saiu e a cabeça ficou dentro da sua guarda.
Human: Como fecho o triângulo direito?
AI: Ajuste o ângulo, puxe o braço dele cruzando e trave a perna atrás do joelho.
Human: Ele levantou e ficou em pé na minha guar 

===== resumo+janela: 262 tokens | menciona joelho: True
System: The human has a left‑knee injury

In [ ]:
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", MESTRE_SYSTEM),
    ("system", "Memory of the conversation so far:\n{history}"),
    ("human", "{question}"),
]).partial(format_instructions=parser.get_format_instructions())

memory_chain = memory_prompt | llm | parser

results = {}
for name, text in contexts.items():
    resp = memory_chain.invoke({"question": final_question, "history": text})
    answer = to_text(resp) + " " + (resp.safety or "")
    results[name] = answer
    print(f"===== {name} | cita joelho: {'joelho' in answer.lower()}\n{answer[:350]}\n")
    time.sleep(10)


===== buffer | cita joelho: True
Você está na guarda aberta e quer uma raspagem que proteja o joelho esquerdo, evitando torções ou pressão excessiva nessa perna. Steps: 1. Pegue a manga direita do adversário com a mão direita e a gola esquerda com a mão esquerda, mantendo a pegada firme. | 2. Execute um arm‑drag: puxe o braço direito dele para fora, criando um espaço e fazendo com

===== janela | cita joelho: True
Você está na guarda aberta e quer uma raspagem segura que funcione contra a maioria dos oponentes. Steps: Segure o tornozelo do oponente com a mão do mesmo lado, puxando‑o para perto da sua perna. | Coloque o pé da mesma perna na parte interna da coxa dele, próximo ao joelho, para bloquear o movimento. | Com a outra perna, faça um gancho (hook) na 

===== resumo+janela | cita joelho: True
Raspagem segura da guarda aberta para um praticante de kimono com lesão no joelho esquerdo, usando o gancho De La Riva sem torção do joelho. Steps: Estabeleça controle da lapela com a mão op

In [ ]:
INJURY_TERMS = ["lesão", "lesionad", "joelho esquerdo", "injur", "left knee", "left-knee", "left‑knee"]

def mentions_injury(text: str) -> bool:
    t = text.lower()
    return any(term in t for term in INJURY_TERMS)

print(f"{'memória':14} {'tokens':>6}  lesão no contexto  resposta considera")
for name in contexts:
    print(f"{name:14} {len(enc.encode(contexts[name])):>6}  {str(mentions_injury(contexts[name])):17}  {mentions_injury(results[name])}")

memória        tokens  lesão no contexto  resposta considera
buffer            288  True               True
janela            146  False              False
resumo+janela     262  True               True
vetorial           77  False              False


## 3.5 Reconhecimento de intenção + extração de entidades

Por turno: `intent_chain` (v2 da fase 2) e `entity_chain` rodam em paralelo → entidades atualizam a memória de trabalho (`FightState`) e o perfil (`StudentProfile`) → os dois entram no prompt do Mestre junto com a janela curta da conversa.


In [29]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableParallel
from src.tools import search_video

Belt = Literal["white", "blue", "purple", "brown", "black"]

class BJJEntities(BaseModel):
    position: str | None = Field(None, description="Position in English, e.g. 'closed guard', 'mount', 'open guard'")
    technique: str | None = Field(None, description="Technique being discussed, in English, e.g. 'armlock', 'scissor sweep'")
    opponent_action: str | None = Field(None, description="What the opponent did, in English, e.g. 'pinned my arm', 'stood up'")
    belt: Belt | None = None
    gi: Literal["gi", "nogi"] | None = Field(None, description="gi = kimono; nogi = sem kimono")
    injuries: list[str] = Field(default_factory=list, description="Injuries the student reports, in English, e.g. 'left knee'")

class FightState(BaseModel):          # memória de trabalho
    position: str | None = None
    technique: str | None = None
    opponent_action: str | None = None

class StudentProfile(BaseModel):      # memória semântica
    belt: Belt | None = None
    gi: Literal["gi", "nogi"] | None = None
    injuries: list[str] = Field(default_factory=list)
    studied_techniques: list[str] = Field(default_factory=list)

entity_parser = PydanticOutputParser(pydantic_object=BJJEntities)

entity_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Extract BJJ entities from the student's LATEST message only.\n"
     "Use the current state only to resolve references like 'isso', 'ele', 'daí'.\n"
     "Leave a field null (or the list empty) when the latest message does not mention it. Never copy fields from the state.\n"
     "Current state: {state}\n\n{format_instructions}"),
    ("human", "{question}"),
]).partial(format_instructions=entity_parser.get_format_instructions())

entity_chain = entity_prompt | llm_small | entity_parser   # extração: 20b basta

understand = RunnableParallel(info=intent_chain, ents=entity_chain)

def reset_memory():
    global profile, state, short_mem
    profile, state = StudentProfile(), FightState()
    short_mem = ConversationBufferMemory(memory_key="history", return_messages=True)


In [30]:
def update_memory(
    profile: StudentProfile, state: FightState, ents: BJJEntities
) -> None:
    """Regra de merge: campo novo não-nulo sobrescreve; listas acumulam sem repetir."""
    for field in ("position", "technique", "opponent_action"):
        if getattr(ents, field):
            setattr(state, field, getattr(ents, field))
    for field in ("belt", "gi"):
        if getattr(ents, field):
            setattr(profile, field, getattr(ents, field))
    for injury in ents.injuries:
        if injury not in profile.injuries:
            profile.injuries.append(injury)
    if ents.technique and ents.technique not in profile.studied_techniques:
        profile.studied_techniques.append(ents.technique)


answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", MESTRE_SYSTEM),
        (
            "system",
            "Student profile (long-term memory): {profile}\n"
            "Current situation (working memory): {state}\n"
            "Use the profile: the belt sets the level, and NEVER suggest techniques that load an injured body part.",
        ),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
).partial(format_instructions=parser.get_format_instructions())

answer_chain = answer_prompt | llm | parser

condense_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the student's LATEST message as ONE standalone question, in the student's language.\n"
            "Resolve references ('isso', 'disso', 'ele', 'daí', 'agora') using the recent conversation and the current situation.\n"
            "Always name the current position and technique from the situation when the message depends on them. Do not answer and do not add new facts.\n"
            "If the message is already standalone, return it unchanged.\n\n"
            "Current situation: {state}\nRecent conversation:\n{history}",
        ),
        ("human", "{question}"),
    ]
)
condense_chain = condense_prompt | llm_small | StrOutputParser()

profile, state = StudentProfile(), FightState()
short_mem = ConversationBufferMemory(memory_key="history", return_messages=True)


def mestre_turn(question: str, config: dict | None = None) -> dict:
    history = trim_messages(
        short_mem.load_memory_variables({})["history"],
        strategy="last",
        max_tokens=300,
        token_counter=count_tokens,
        start_on="human",
    )
    # 1º turno não tem o que resolver
    standalone = (
        condense_chain.invoke(
            {
                "question": question,
                "state": state.model_dump_json(exclude_none=True),
                "history": get_buffer_string(history),
            },
            config=config,
        )
        if history
        else question
    )

    u = understand.invoke(
        {"question": standalone, "state": state.model_dump_json()}, config=config
    )
    update_memory(profile, state, u["ents"])

    if u["info"].intent == "video":
        # vídeo não precisa do 120b: a busca já é a resposta
        term = (
            u["info"].search_term
            or f"{state.technique or ''} {state.position or ''} bjj"
        )
        video = search_video.invoke(term)
        level = "BEGINNER" if profile.belt in (None, "white") else "ADVANCED"
        resp = MestreResponse(level=level, context=f"Vídeo sugerido: {video}")
    else:
        video = None
        resp = answer_chain.invoke(
            {
                "question": question,
                "history": history,
                "profile": profile.model_dump_json(exclude_none=True),
                "state": state.model_dump_json(exclude_none=True),
            },
            config=config,
        )

    short_mem.save_context({"input": question}, {"output": to_text(resp)})
    return {
        "question": question,
        "standalone": standalone,
        "intent": u["info"].intent,
        "entities": u["ents"].model_dump(exclude_none=True),
        "state": state.model_dump(exclude_none=True),
        "profile": profile.model_dump(exclude_none=True),
        "answer": resp.model_dump(exclude_none=True),
        "video": video,
    }

In [31]:
turns_35 = [
    "Sou faixa azul, treino com kimono e tenho lesão no joelho esquerdo.",
    "Estou na guarda fechada e ele prendeu meu braço no chão.",
    "E se ele levantar e ficar em pé?",
    "Que raspagem é segura pra mim agora?",
    "Me mostra um vídeo disso.",
]

log_35 = []
for q in turns_35:
    out = mestre_turn(q)
    log_35.append(out)
    print(f">>> {q}")
    print(f"    reescrita: {out['standalone']}")
    print(f"    intent={out['intent']} | ents={out['entities']}")
    print(f"    state={out['state']}")
    print(f"    profile={out['profile']}")
    print(f"    Mestre: {(out['answer'].get('clarifying_question') or out['answer']['context'])[:200]}")
    if out["video"]:
        print(f"    vídeo: {out['video']}")
    print()
    time.sleep(12)  # 3 chamadas de LLM por turno


>>> Sou faixa azul, treino com kimono e tenho lesão no joelho esquerdo.
    reescrita: Sou faixa azul, treino com kimono e tenho lesão no joelho esquerdo.
    intent=off_topic | ents={'belt': 'blue', 'gi': 'gi', 'injuries': ['left knee']}
    state={}
    profile={'belt': 'blue', 'gi': 'gi', 'injuries': ['left knee'], 'studied_techniques': []}
    Mestre: Qual técnica, posição ou conceito você gostaria de trabalhar, levando em conta a lesão no joelho esquerdo?



KeyboardInterrupt: 

## 3.6 Teste multi-turno com trace

Conversa fixa de 6 turnos rodada dentro de um único trace no LangSmith (`@traceable`): cada turno aparece como filho, com reescrita, intenção, entidades e resposta. Evidência dupla: link do trace + transcript JSON em `docs/evidencias/fase3_multiturno.json`.


In [32]:
import json
from pathlib import Path
from langsmith import traceable, get_current_run_tree

turns_36 = turns_35 + ["O que você já sabe sobre mim?"]   # 6º turno: lembrança do perfil


@traceable(name="fase3-multiturno", tags=["fase3"], metadata={"session_id": "aluno-01"})
def run_conversation(turns: list[str]) -> dict:
    reset_memory()
    log = []
    for q in turns:
        log.append(mestre_turn(q))
        time.sleep(12)
    return {"run_id": str(get_current_run_tree().id), "turns": log}


result_36 = run_conversation(turns_36)

for i, t in enumerate(result_36["turns"], 1):
    print(f"{i}. {t['question']}\n   -> {t['standalone']}\n   intent={t['intent']} | state={t['state']}")
    print(f"   Mestre: {(t['answer'].get('clarifying_question') or t['answer']['context'])[:180]}\n")

out_dir = Path("../docs/evidencias")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "fase3_multiturno.json").write_text(
    json.dumps(result_36, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("run_id:", result_36["run_id"])


1. Sou faixa azul, treino com kimono e tenho lesão no joelho esquerdo.
   -> Sou faixa azul, treino com kimono e tenho lesão no joelho esquerdo.
   intent=off_topic | state={}
   Mestre: Qual técnica, posição ou conceito você gostaria de trabalhar, levando em conta a lesão no joelho esquerdo?

2. Estou na guarda fechada e ele prendeu meu braço no chão.
   -> Como escapar quando estou na guarda fechada e o oponente prendeu meu braço no chão?
   intent=technique | state={'position': 'closed guard', 'opponent_action': 'gripped my arm on the ground'}
   Mestre: Você está na guarda fechada e o adversário prendeu seu braço no chão; o objetivo é criar espaço e finalizar ou varrer sem sobrecarregar o joelho esquerdo.

3. E se ele levantar e ficar em pé?
   -> E se o adversário levantar e ficar em pé na guarda fechada?
   intent=technique | state={'position': 'closed guard', 'opponent_action': 'lift up and stand up'}
   Mestre: O oponente se levanta da sua guarda fechada e fica em pé; o objetiv